# NYC Substations

| | |
|---|---|
| **Source** | OpenStreetMap (OSM) |
| **URL** | [openstreetmap.org](https://www.openstreetmap.org) |
| **Granularity** | One row per mapped `power=substation` feature |
| **Data collection** | Volunteered geographic information; no official publisher or data dictionary |

NYC power substations are not published as an open dataset by ConEdison or any city agency; grid infrastructure location is treated as a national security concern and is not released with a formal schema. This dataset is instead derived from OpenStreetMap, a community-maintained map where contributors tag real-world features with free-form key/value pairs. There is no authoritative data dictionary for these tags: descriptions below follow the informal conventions documented on the [OSM Wiki](https://wiki.openstreetmap.org/wiki/Tag:power=substation), and coverage/consistency depends entirely on what volunteers have mapped.

## Column Reference

OSM tags are optional and inconsistently populated. The columns below are the tags retained after processing; any tag not listed was dropped as out of scope for this baseline, not necessarily because it lacks value.

| Column | Description | Notes |
|---|---|---|
| `id` | OSM feature identifier (node/way) | Stable within an OSM snapshot, not a ConEdison asset ID |
| `name` | Human-assigned substation name | Frequently missing; not all substations are named by contributors |
| `power` | OSM feature category | Constant `substation` by construction of the query tags |
| `operator` | Free-text operator tag | Used to scope this dataset to `Consolidated Edison`; volunteered, not verified against utility records |
| `substation` | Substation function/voltage class (e.g. `transmission`, `distribution`) | Informal OSM subtag, not a controlled vocabulary |
| `voltage` | Voltage level(s), semicolon-separated when multiple | Free text; units and formatting are not enforced |
| `frequency` | Grid frequency in Hz | Usually `60` for NYC when present |
| `geometry` | Feature location | Original OSM geometry may be a `Point`, `Polygon`, or other type depending on how the feature was mapped |

## Data Acquisition

In [1]:
import json
import requests
import folium
from datetime import date
from pathlib import Path
import pandas as pd
import geopandas as gpd
import osmnx as ox

In [2]:
ROOT = Path("..").resolve()
BOUNDARIES_GEOJSON_PATH = ROOT / "data/raw/borough_boundaries.geojson"
PARAMS = {
    "tags": {"power": "substation"},
}
GEOJSON_PATH = ROOT / "data/raw/osm_substations.geojson"
META_GEOJSON_PATH = ROOT / "data/raw/osm_substations.geojson.meta.json"

# Fetch NYC power substations from OpenStreetMap inside borough boundaries
boroughs = gpd.read_file(BOUNDARIES_GEOJSON_PATH)
nyc_polygon = boroughs.union_all()

ox.settings.max_query_area_size = 5e9
ox.settings.cache_folder = "/tmp/osm_cache"

fetch_gdf = (
    gpd.read_file(GEOJSON_PATH)
    if GEOJSON_PATH.exists()
    else ox.features_from_polygon(nyc_polygon, tags=PARAMS["tags"])
)

if GEOJSON_PATH.exists():
    print(f"Already fetched → {GEOJSON_PATH}, skipping.")
else:
    GEOJSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    fetch_gdf.to_file(GEOJSON_PATH, driver="GeoJSON")
    META_GEOJSON_PATH.write_text(
        json.dumps(
            {
                "fetched_date": str(date.today()),
                "source_url": "https://www.openstreetmap.org",
                "source_params": PARAMS,
                "row_count": len(fetch_gdf),
            },
            indent=2,
        )
    )
    print(f"Saved {GEOJSON_PATH.name} ({META_GEOJSON_PATH.read_text()})")
    
gpd.read_file(GEOJSON_PATH)

Already fetched → /home/henrique/agentwin-dataset/data/raw/osm_substations.geojson, skipping.


,element,id,addr:state,ele,gnis:feature_id,name,power,source,barrier,check_date,...,website,voltage:primary,voltage:secondary,description,abandoned,protection_title,source:1,source_ref:1,start_date,geometry
0,node,368062840,NY,2,2062029,Port Morris Power Station,substation,USGS Geonames,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-73.90291 40.80455)
1,way,108298302,NaN,NaN,NaN,Rainey Substation,substation,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-73.94341 40.76417, -73.94344 40.764..."
2,way,108318934,NaN,NaN,NaN,NaN,substation,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-73.91051 40.78725, -73.90981 40.786..."
3,way,108322562,NaN,NaN,NaN,Vernon Substation,substation,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-73.94735 40.75878, -73.94591 40.758..."
4,way,108331538,NaN,NaN,NaN,North Queens Substation,substation,NaN,fence,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-73.90605 40.78116, -73.90581 40.781..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199,way,1485150724,NaN,NaN,NaN,NaN,substation,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-73.89792 40.78213, -73.89807 40.782..."
200,way,1485150725,NaN,NaN,NaN,NaN,substation,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-73.91294 40.78657, -73.91255 40.786..."
201,way,1485150726,NaN,NaN,NaN,NaN,substation,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-73.91279 40.78606, -73.91258 40.786..."
202,way,1508559019,NaN,NaN,NaN,NaN,substation,NaN,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-74.01227 40.7138, -74.01241 40.7132..."


In [3]:
CSV_PATH = ROOT / "data/raw/osm_substations.csv"
META_CSV_PATH = ROOT / "data/raw/osm_substations.csv.meta.json"

if CSV_PATH.exists():
    print(f"Already fetched → {CSV_PATH}, skipping.")
else:
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    fetch_gdf.to_csv(CSV_PATH, index=False)
    META_CSV_PATH.write_text(
        json.dumps(
            {
                "fetched_date": str(date.today()),
                "source_url": "https://www.openstreetmap.org",
                "source_params": PARAMS,
                "row_count": len(fetch_gdf),
            },
            indent=2,
        )
    )
    print(f"Saved {CSV_PATH.name} ({META_CSV_PATH.read_text()})")
    
pd.read_csv(CSV_PATH)


Already fetched → /home/henrique/agentwin-dataset/data/raw/osm_substations.csv, skipping.


,geometry,addr:state,ele,gnis:feature_id,name,power,source,barrier,check_date,fixme,...,building:part,website,voltage:primary,voltage:secondary,description,abandoned,protection_title,source:1,source_ref:1,start_date
0,POINT (-73.9029142 40.8045454),NY,2.0,2062029.0,Port Morris Power Station,substation,USGS Geonames,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"POLYGON ((-73.9434133 40.7641675, -73.9434382 ...",NaN,NaN,NaN,Rainey Substation,substation,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"POLYGON ((-73.9105054 40.7872511, -73.909806 4...",NaN,NaN,NaN,NaN,substation,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"POLYGON ((-73.9473458 40.7587815, -73.945906 4...",NaN,NaN,NaN,Vernon Substation,substation,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"POLYGON ((-73.9060539 40.7811609, -73.9058098 ...",NaN,NaN,NaN,North Queens Substation,substation,NaN,fence,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199,"POLYGON ((-73.8979161 40.7821327, -73.8980663 ...",NaN,NaN,NaN,NaN,substation,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200,"POLYGON ((-73.9129445 40.7865672, -73.9125489 ...",NaN,NaN,NaN,NaN,substation,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,"POLYGON ((-73.9127876 40.7860554, -73.9125811 ...",NaN,NaN,NaN,NaN,substation,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
202,"POLYGON ((-74.0122726 40.7137959, -74.0124137 ...",NaN,NaN,NaN,NaN,substation,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Processing

The raw OSM extract requires five cleaning steps before it can be used as the baseline substation dataset. Because OSM has no formal schema, these steps favor explicit, documented decisions over blanket missing-value heuristics.

1. **Keep relevant columns** - Most OSM tags on this extract (e.g. `source`, `element`, `location`, `operator:wikidata`) are sparse, contributor-dependent metadata not needed for the current scenario; they remain available in the raw snapshot for provenance.
2. **Scope to Consolidated Edison** - The dataset is restricted to features whose `operator` tag equals `Consolidated Edison`. This reflects OSM tagging, not a verified utility service-territory boundary; substations with a missing or differently formatted operator tag are excluded even if they may belong to ConEdison.
3. **Require a unique, non-null id** - The OSM `id` is the only stable identifier available and is used to drop untagged and duplicate features.
4. **Validate geometry** - Geometries are confirmed non-null, non-empty, valid, and expressed in WGS84 (EPSG:4326), the implicit CRS of GeoJSON per RFC 7946.
5. **Derive a representative point** - OSM substations may be mapped as points, polygons, or other geometry types depending on how contributors surveyed them. The original geometry type is preserved in `original_geometry_type`, and a single representative point is derived for consistent downstream use.

In [ ]:
gdf = gpd.read_file(GEOJSON_PATH)
print(f"Fetched {len(gdf)} rows from {GEOJSON_PATH.name}")

# 1. Filter to only include relevant columns
columns = [
    "id",
    "name",
    "power",
    "operator",
    "substation",
    "voltage",
    "frequency",
    "geometry",
]

gdf = gdf[[column for column in columns if column in gdf.columns]]

# 2. Restrict the baseline to OSM features tagged as ConEdison-operated.
gdf = gdf.loc[
    gdf["operator"].eq("Consolidated Edison")
].copy()

# 3. Require a non-null, unique OSM identifier.
gdf = gdf.dropna(subset=["id"]).drop_duplicates(subset=["id"])

# 4. Ensure a defined WGS84 geometry suitable for point derivation.
assert gdf.crs is not None, "GeoDataFrame has no CRS defined."
gdf = gdf.to_crs(epsg=4326)

geometry_mask = (
    gdf.geometry.notna()
    & ~gdf.geometry.is_empty
    & gdf.geometry.is_valid
)
gdf = gdf.loc[geometry_mask].copy()

# 5. Preserve source geometry type and derive representative locations.
gdf["original_geometry_type"] = gdf.geometry.geom_type
gdf = gdf.set_geometry(gdf.geometry.representative_point())

# Write to file
gdf.to_file(ROOT / "data/processed/substations.geojson", driver="GeoJSON")
gdf.to_csv(ROOT / "data/processed/substations.csv")


print(f"Processed {len(gdf)} rows from {GEOJSON_PATH.name} and saved to processed files.")

gdf

Fetched 204 rows from osm_substations.geojson
Processed 104 rows from osm_substations.geojson and saved to processed files.


,id,name,power,operator,substation,voltage,frequency,geometry,original_geometry_type
1,108298302,Rainey Substation,substation,Consolidated Edison,transmission,345000;138000,60,POINT (-73.94301 40.76329),Polygon
3,108322562,Vernon Substation,substation,Consolidated Edison,distribution,138000,60,POINT (-73.94692 40.75802),Polygon
4,108331538,North Queens Substation,substation,Consolidated Edison,distribution,138000;27000,60,POINT (-73.90674 40.78177),Polygon
6,187984305,Jamaica Substation,substation,Consolidated Edison,transmission,138000;27000,60,POINT (-73.81391 40.69919),Polygon
8,188367030,West 49th Street Substation,substation,Consolidated Edison,transmission,345000,60,POINT (-73.99597 40.76604),Polygon
...,...,...,...,...,...,...,...,...,...
191,1338654396,11th Street Conduit,substation,Consolidated Edison,NaN,NaN,NaN,POINT (-73.95241 40.73839),Polygon
192,1338654415,11th Street Conduit,substation,Consolidated Edison,yes,NaN,NaN,POINT (-73.95075 40.74406),Polygon
195,1434668163,Woodrow Substation,substation,Consolidated Edison,distribution,138000;13200,60,POINT (-74.21525 40.5548),Polygon
196,1472076552,Willowbrook Substation,substation,Consolidated Edison,distribution,138000;33000;13200;4000,50,POINT (-74.14564 40.61078),Polygon


## Visualization